# Projeto 3 — Resultados dos seis treinamentos

**Autores:** Paulo Henrico Rabelo (2312652) e Joey Alan (2320416)

Lê apenas os arquivos gerados por `scripts/treinar.py`, `scripts/prever.py` e `scripts/avaliar.py`; nada é treinado aqui.
O teste só aparece depois de `scripts/prever.py --teste`, executado uma vez ao fim das escolhas.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as Show, display
from PIL import Image

from projeto3.classes import CLASSES, NOMES_PT

FIG = ROOT / "resultados" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)
FONTE = "Fonte: elaborado pelos autores a partir do NWPU-RESISC45 (Cheng; Han; Lu, 2017)."
pd.set_option("display.width", 140)


def salvar(fig, nome):
    fig.text(0.01, 0.003, FONTE, fontsize=7, color="0.35")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(FIG / nome, dpi=150)
    plt.show()

experimentos = pd.read_csv(ROOT / "resultados/experimentos.csv")
arquiteturas = pd.read_csv(ROOT / "resultados/arquiteturas.csv")
tem_teste = "f1_macro_teste" in experimentos
print("Teste avaliado:", tem_teste)

## Arquiteturas e parâmetros

In [ ]:
for experimento in arquiteturas["experimento"]:
    print((ROOT / "resultados/arquiteturas" / f"{experimento}.txt").read_text(encoding="utf-8"))
arquiteturas

## Desempenho e custo

In [ ]:
colunas = ["experimento", "parametros", "tamanho_modelo_mb", "lote", "epocas_executadas", "checkpoint_epoca",
           "tempo_total_s", "tempo_medio_epoca_s", "dispositivo", "acuracia_validacao", "f1_macro_validacao"]
if tem_teste:
    colunas += ["acuracia_teste", "f1_macro_teste", "precisao_macro_teste", "recall_macro_teste", "classes_sem_previsao_teste"]
experimentos[colunas]

In [ ]:
alvo = "teste" if tem_teste else "validacao"
display(Show(FIG / f"desempenho_custo_{alvo}.png"))

## Curvas de aprendizagem (checkpoint marcado em vermelho)

In [ ]:
display(Show(FIG / "painel_curvas_perda.png"))
display(Show(FIG / "painel_curvas_acuracia.png"))
for experimento in experimentos["experimento"]:
    display(Show(FIG / f"curvas_{experimento}.png"))

## Métricas por classe

In [ ]:
for experimento in experimentos["experimento"]:
    tabela = pd.read_csv(ROOT / "resultados/metricas" / f"{experimento}_{alvo}_por_classe.csv")
    print(f"\n{experimento} ({alvo})")
    display(tabela.round(3))
display(Show(FIG / f"f1_por_classe_{alvo}.png"))

## Matrizes de confusão (reais nas linhas, previstas nas colunas)

In [ ]:
display(Show(FIG / f"painel_confusao_{alvo}.png"))
for experimento in experimentos["experimento"]:
    display(Show(FIG / f"confusao_{experimento}_{alvo}.png"))

In [ ]:
# Pares de classes mais confundidos (fora da diagonal), somando os seis modelos
total = None
for experimento in experimentos["experimento"]:
    m = pd.read_csv(ROOT / "resultados/metricas" / f"{experimento}_{alvo}_confusao.csv", index_col=0)
    total = m if total is None else total + m
fora = [(total.loc[r, p], r, p) for r in CLASSES for p in CLASSES if r != p]
pd.DataFrame(sorted(fora, reverse=True)[:10], columns=["erros", "real", "prevista"])

## Análise de erros (modelo de maior F1 macro de validação)

In [ ]:
if tem_teste:
    print(json.loads((ROOT / "resultados/selecao.json").read_text(encoding="utf-8")))
    display(pd.read_csv(ROOT / "resultados/erros_teste.csv"))
    display(Show(FIG / "erros_teste.png"))
    if (FIG / "erros_teste_YOLO-cls.png").exists():
        display(Show(FIG / "erros_teste_YOLO-cls.png"))
else:
    print("Teste ainda não avaliado.")

## Roteiro da discussão (para o artigo)

1. Como MLP A/B e CNN A/B se comportaram em 128×128? A profundidade ajudou a MLP? O viés indutivo da convolução compensou?
2. Como a resolução 256×256 afetou cada CNN (desempenho, tempo por época, memória)? O campo receptivo fixo cobre metade da
   cena em 256 px — isso favorece a CNN B, mais profunda?
3. Os ganhos compensaram o custo (tempo total, tempo/época, parâmetros, tamanho do arquivo)?
4. Quais classes foram confundidas e quais aspectos visuais explicam (água/areia, textura de relevo, estruturas circulares)?
5. Curvas: há subajuste (perdas altas e próximas) ou sobreajuste (treino cai, validação sobe)? Em que época?
6. Limites de generalização: um único treino por configuração, sensores/regiões/estações diferentes, resolução espacial
   variável do RESISC45 (≈0,2 a 30 m/pixel).